In [ ]:
import glob
import os
from pathlib import Path

import autoroot
import hydra
import matplotlib.pyplot as plt
import numpy as np
import scipy
import torch
from omegaconf import OmegaConf

from src.data.lf.lf_datamodule import LF
from src.modules.layers.ssm import diagSSM
from src.modules.lit_module import LitModule
from src.modules.models.sequential import SequentialModel
from src.utils import initialize_resolvers
from src.utils.tools import get_rho_lstsq

initialize_resolvers()

In [ ]:
def data_illustration(inputs, outputs):

    fig, ax = plt.subplots(1, 2, figsize=(15, 5))

    ax[0].plot(inputs[0, :, 0].numpy())
    ax[0].plot(outputs[0, :, 0].numpy())

    # ax[1].plot(get_rho_lstsq(inputs, outputs, end_point=False)[:, 0])

In [ ]:
data1 = LF(
    data_dir="",
    N=5,
    T=10000,
    D=1,
    rho_name="impulse",
    rho_param={"delta": 5000},
    # input_generator="gaussian",
).dataset_construction()
# data1 = LF(
#     data_dir="",
#     N=50,
#     T=1024,
#     D=1,
#     rho_name="airy",
#     rho_param={"c": 10},
#     input_generator="gaussian",
#     dt=0.01
# ).dataset_construction()
data_illustration(data1[:][0], data1[:][1])

In [ ]:
path = "./logs/train/2024-05-10_18-37-21"
path = Path(path)
ckpt = max((path / "checkpoints").glob("*.ckpt"), key=os.path.getctime)

ckpt_init = path / "checkpoints" / "initial_checkpoint.ckpt"
model_init = LitModule.load_from_checkpoint(ckpt_init)
model = LitModule.load_from_checkpoint(ckpt)
x = data1[:][0]
y = data1[:][1]
yhat = model(x.to("cuda:0")).cpu().detach()
data_illustration(data1[:][0], yhat)

In [ ]:
At = model.model.layers[0]._A
Bt = model.model.layers[0].B
Ct = model.model.layers[0].C

A = model.model.layers[0]._A.data.cpu().numpy()
B = model.model.layers[0].B.data.cpu().numpy()
C = model.model.layers[0].C.data.cpu().numpy()

In [ ]:
from src.functionals.vandermonde import *

In [ ]:
log_vandermonde_naive(Bt.permute(2, 0, 1) * Ct.permute(1, 0, 2), At * 1j, L=10, conj=False)

In [ ]:
log_vandermonde_lazy(Bt.permute(2, 0, 1) * Ct.permute(1, 0, 2), At * 1j, L=10)

In [ ]:
torch.sum(log_vandermonde(Bt.permute(2, 0, 1) * Ct.permute(1, 0, 2), At * 1j, L=10), axis=1).shape

In [ ]:
d_model = At.shape[0]
K = 0
for i in range(d_model):
    vi = Bt[i].T * Ct[i]
    Ki = log_vandermonde_naive(vi, At[i] * 1j, 10, conj=False)
    K += Ki
K

In [ ]:
import torch

# Define a float tensor A
A = torch.tensor([1.0, 2.0, 3.0], dtype=torch.float32)

# Multiply A by 1j
result = A * 1j

# Check the type of the result
print(result.dtype)